In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
import os
from pathlib import Path

C:\Users\Administrator\AppData\Local\Temp\ipykernel_46668\1913730594.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [2]:
names = locals()

In [ ]:
pseudo_codes_economies_142 = pd.read_csv(RESULTS / 'pseudo-codes-economies-142.csv')

In [ ]:
def distribute_national_demand_to_pixel(material, start, end, scenario, res='50km'):
    future = np.linspace(2023, 2050, 28)
    economy_raster_ = rio.open(RESULTS / 'economy-rasters' / f'economy-raster-eck4-{res}.tif')
    economy_raster = economy_raster_.read(1)

    input_path_1 = os.path.join('..', 'results', 'demand-rasters', f'global-building-minerals-{int(end-5)}-{res}.tif')
    input_path_2 = os.path.join('..', 'results', 'demand-rasters', f'global-building-minerals-{end}-{res}.tif')

    global_building_minerals_start = rio.open(input_path_1).read(1)
    global_building_minerals_end = rio.open(input_path_2).read(1)

    demand_raster_for_stock_growth_l = np.zeros(shape=economy_raster_.shape, dtype="float32")
    demand_raster_for_stock_growth_m = np.zeros(shape=economy_raster_.shape, dtype="float32")
    demand_raster_for_stock_growth_u = np.zeros(shape=economy_raster_.shape, dtype="float32")
    demand_raster_for_stock_turnover_l = np.zeros(shape=economy_raster_.shape, dtype="float32")
    demand_raster_for_stock_turnover_m = np.zeros(shape=economy_raster_.shape, dtype="float32")
    demand_raster_for_stock_turnover_u = np.zeros(shape=economy_raster_.shape, dtype="float32")

    with tqdm(range(1, 143), desc=f'distributing of {material} demand in {start}-{end} of {scenario}') as pbar:
        for i in pbar:
            economy = pseudo_codes_economies_142[pseudo_codes_economies_142['value'] == i]['economies_142'].values[0]
            pbar.set_postfix_str(f'country={economy}')
            
            path_1 = f'../results/material-flows/future-uncertainty/{economy}/{scenario}/{material}_extraction_for_stock_turnover.npz'
            path_2 = f'../results/material-flows/future-uncertainty/{economy}/{scenario}/{material}_extraction_for_stock_growth.npz'

            value_array_1 = np.load(path_1)['array']
            value_array_2 = np.load(path_2)['array']

            df_extraction_1 = pd.DataFrame(np.percentile(value_array_1, [2.5, 50, 97.5], axis=0).T, columns=['lower', 'mid', 'upper'], index=future)
            df_extraction_2 = pd.DataFrame(np.percentile(value_array_2, [2.5, 50, 97.5], axis=0).T, columns=['lower', 'mid', 'upper'], index=future)

            extraction_for_stock_turnover_l = df_extraction_1.loc[start:end, 'lower'].sum()
            extraction_for_stock_turnover_m = df_extraction_1.loc[start:end, 'mid'].sum()
            extraction_for_stock_turnover_u = df_extraction_1.loc[start:end, 'upper'].sum()
            extraction_for_stock_growth_l = df_extraction_2.loc[start:end, 'lower'].sum()
            extraction_for_stock_growth_m = df_extraction_2.loc[start:end, 'mid'].sum()
            extraction_for_stock_growth_u = df_extraction_2.loc[start:end, 'upper'].sum()

            mask_1 = economy_raster == i
            mask_2 = (economy_raster == i) & (global_building_minerals_end > global_building_minerals_start)

            total_1 = np.sum(global_building_minerals_start[mask_1])
            total_2 = np.sum(global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2])

            demand_raster_for_stock_turnover_l[mask_1] = 1e3 * extraction_for_stock_turnover_l * global_building_minerals_start[mask_1] / total_1
            demand_raster_for_stock_turnover_m[mask_1] = 1e3 * extraction_for_stock_turnover_m * global_building_minerals_start[mask_1] / total_1
            demand_raster_for_stock_turnover_u[mask_1] = 1e3 * extraction_for_stock_turnover_u * global_building_minerals_start[mask_1] / total_1

            demand_raster_for_stock_growth_l[mask_2] = 1e3 * extraction_for_stock_growth_l * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
            demand_raster_for_stock_growth_m[mask_2] = 1e3 * extraction_for_stock_growth_m * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
            demand_raster_for_stock_growth_u[mask_2] = 1e3 * extraction_for_stock_growth_u * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
        
    demand_raster_l = demand_raster_for_stock_turnover_l + demand_raster_for_stock_growth_l
    demand_raster_m = demand_raster_for_stock_turnover_m + demand_raster_for_stock_growth_m
    demand_raster_u = demand_raster_for_stock_turnover_u + demand_raster_for_stock_growth_u

    output_path_7 = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-{start}-{end}-{scenario}-{res}-L.tif')
    output_path_8 = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-{start}-{end}-{scenario}-{res}-M.tif')
    output_path_9 = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-{start}-{end}-{scenario}-{res}-H.tif')

    raster_set = [
        demand_raster_l, 
        demand_raster_m, 
        demand_raster_u
        ]
    
    output_path_set = [
        output_path_7, 
        output_path_8, 
        output_path_9
        ]

    for array, output_path in zip(raster_set, output_path_set):
        with rio.open(
            output_path,
            "w",
            driver="GTiff",
            height=economy_raster_.shape[0],
            width=economy_raster_.shape[1],
            count=1,
            dtype=np.float32,
            crs=economy_raster_.crs,
            transform=economy_raster_.transform,
            compress='lzw'
        ) as dst:
            dst.write(array, 1)


In [5]:
def add_demand_together(material, scenario, res='50km'):
    country_raster_ = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')
    country_raster = country_raster_.read(1)
    total_demand_raster_L = np.zeros(shape=country_raster_.shape, dtype="float32")
    total_demand_raster_M = np.zeros(shape=country_raster_.shape, dtype="float32")
    total_demand_raster_U = np.zeros(shape=country_raster_.shape, dtype="float32")

    for start, end in zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050]):
        input_path_l = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-{start}-{end}-{scenario}-{res}-L.tif')
        input_path_m = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-{start}-{end}-{scenario}-{res}-M.tif')
        input_path_u = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-{start}-{end}-{scenario}-{res}-H.tif')

        total_demand_raster_L += rio.open(input_path_l).read(1)
        total_demand_raster_M += rio.open(input_path_m).read(1)
        total_demand_raster_U += rio.open(input_path_u).read(1)
    
    global_l = np.sum(total_demand_raster_L) / 1e9
    global_m = np.sum(total_demand_raster_M) / 1e9
    global_u = np.sum(total_demand_raster_U) / 1e9

    print(f'{material} ({scenario}): {global_m:.0f} ({global_l:.0f}-{global_u:.0f}) Gt')

    output_path_l = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-2023-2050-{scenario}-{res}-L.tif')
    output_path_m = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-2023-2050-{scenario}-{res}-M.tif')
    output_path_u = os.path.join('..', 'results', 'uncertainty-analysis', f'{material}-demand-2023-2050-{scenario}-{res}-H.tif')

    for total_demand_raster, output_path in zip(
        [total_demand_raster_L, total_demand_raster_M, total_demand_raster_U], 
        [output_path_l, output_path_m, output_path_u]):
        with rio.open(
            output_path,
            "w",
            driver="GTiff",
            height=country_raster_.shape[0],
            width=country_raster_.shape[1],
            count=1,
            dtype=np.float32,
            crs=country_raster_.crs,
            transform=country_raster_.transform,
            compress='lzw'
        ) as dst:
            dst.write(total_demand_raster, 1)

In [6]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']):
            distribute_national_demand_to_pixel(material, start, end, scenario, '50km')

  0%|          | 0/3 [00:00<?, ?it/s]

0it [00:00, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

distributing of sand demand in 2023-2025 of BAU:   0%|          | 0/142 [00:00<?, ?it/s]

distributing of sand demand in 2023-2025 of MIU:   0%|          | 0/142 [00:00<?, ?it/s]

...

distributing of limestone demand in 2046-2050 of RE:   0%|          | 0/142 [00:00<?, ?it/s]

distributing of limestone demand in 2046-2050 of RC:   0%|          | 0/142 [00:00<?, ?it/s]

distributing of limestone demand in 2046-2050 of SS:   0%|          | 0/142 [00:00<?, ?it/s]

distributing of limestone demand in 2046-2050 of CCR:   0%|          | 0/142 [00:00<?, ?it/s]

distributing of limestone demand in 2046-2050 of CE:   0%|          | 0/142 [00:00<?, ?it/s]

In [7]:
for scenario in tqdm(['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']):
    add_demand_together('sand', scenario, res='50km')
    add_demand_together('gravel', scenario, res='50km')
    add_demand_together('limestone', scenario, res='50km')

  0%|          | 0/8 [00:00<?, ?it/s]

sand (BAU): 381 (339-425) Gt
gravel (BAU): 247 (219-277) Gt
limestone (BAU): 161 (113-214) Gt
sand (MIU): 356 (317-398) Gt
gravel (MIU): 231 (205-260) Gt
limestone (MIU): 151 (105-201) Gt
sand (LE): 373 (333-414) Gt
gravel (LE): 242 (216-269) Gt
limestone (LE): 158 (111-209) Gt
sand (RE): 371 (330-414) Gt
gravel (RE): 240 (213-269) Gt
limestone (RE): 157 (110-208) Gt
sand (RC): 361 (322-402) Gt
gravel (RC): 232 (207-261) Gt
limestone (RC): 161 (113-214) Gt
sand (SS): 339 (302-378) Gt
gravel (SS): 247 (219-277) Gt
limestone (SS): 161 (113-214) Gt
sand (CCR): 381 (339-425) Gt
gravel (CCR): 247 (219-277) Gt
limestone (CCR): 139 (97-185) Gt
sand (CE): 287 (257-319) Gt
gravel (CE): 205 (183-229) Gt
limestone (CE): 125 (87-165) Gt
